# Approximate Q-Learning Example

This notebook demonstrates Approximate Q-Learning with a very simple environment.
We use a linear function approximation for the Q-function.

The goal is to show how learning works step by step and how the learned policy is applied.

## Environment Description

The environment is a 1D line with 3 states: 0, 1, 2.
State 2 is a terminal state.

Actions:
Action 0: move left
Action 1: move right

Reward:
Reaching state 2 gives reward +1.
All other transitions give reward 0.

In [1]:
# Định nghĩa các tham số của môi trường
num_states = 3            # số trạng thái
num_actions = 2           # số hành động
terminal_state = 2        # trạng thái kết thúc

# Hàm mô phỏng môi trường
def step(state, action):
    # Nếu đã ở trạng thái kết thúc thì không di chuyển nữa
    if state == terminal_state:
        return state, 0
    
    # Hành động sang trái
    if action == 0:
        next_state = max(0, state - 1)
    # Hành động sang phải
    else:
        next_state = min(num_states - 1, state + 1)
    
    # Phần thưởng
    reward = 1 if next_state == terminal_state else 0
    
    return next_state, reward

## Feature Representation

We approximate Q(s, a) as:

Q(s, a) = w · f(s, a)

where:
- f(s, a) is a feature vector 
- w is the weight vector to be learned.
- The “·” symbol denotes the dot product.

Here we use a very simple feature representation:
One-hot encoding for (state, action).

In [2]:
# Hàm tạo vector đặc trưng f(s, a)
def features(state, action):
    # Khởi tạo vector đặc trưng toàn 0
    f = [0.0 for _ in range(num_states * num_actions)]
    
    # Tính chỉ số tương ứng với cặp (state, action)
    index = state * num_actions + action
    
    # Gán giá trị 1 cho đặc trưng tương ứng
    f[index] = 1.0
    
    return f

## Approximate Q-Function

We compute Q(s, a) = w · f(s, a) as the dot product between weights and features.

In [3]:
# Khởi tạo vector trọng số w với giá trị 0
weights = [0.0 for _ in range(num_states * num_actions)]

# Hàm tính Q(s, a)
def q_value(state, action):
    # Lấy vector đặc trưng
    f = features(state, action)
    
    # Tính tích vô hướng giữa w và f (Q(s, a) = w · f(s, a))
    q = sum(w * x for w, x in zip(weights, f))
    
    return q

## Learning with Approximate Q-Learning

Update rule:

$w = w + α.(r + γ.max_{a'} Q(s', a') − Q(s, a)).f(s, a)$

or:

$w = w + α.td\_error.f(s, a)$

We use epsilon-greedy for action selection.

In [4]:
import random

# Các siêu tham số
alpha = 0.1      # tốc độ học
gamma = 0.9      # hệ số chiết khấu
epsilon = 0.1    # xác suất thăm dò
episodes = 50    # số tập huấn luyện

# Vòng lặp huấn luyện
for episode in range(episodes):
    state = 0  # bắt đầu từ trạng thái 0
    
    while state != terminal_state:
        # Chọn hành động theo epsilon-greedy
        if random.random() < epsilon:
            # Khám phá
            # epsilon = 0.1
            # 10% thời gian agent chọn hành động ngẫu nhiên.
            action = random.choice([0, 1])
        else:
            # Khai thác
            # 90% thời gian agent chọn hành động có Q lớn nhất.
            q_left = q_value(state, 0)
            q_right = q_value(state, 1)
            action = 0 if q_left >= q_right else 1
        
        # Thực hiện hành động trong môi trường
        next_state, reward = step(state, action)
        
        # Tính giá trị mục tiêu
        if next_state == terminal_state:
            target = reward
        else:
            target = reward + gamma * max(
                q_value(next_state, 0),
                q_value(next_state, 1)
            )
        
        # Sai số TD
        td_error = target - q_value(state, action)
        
        # Cập nhật trọng số: w = w + α.td_error.f(s, a)
        f = features(state, action)
        for i in range(len(weights)):
            weights[i] += alpha * td_error * f[i]
        
        # Chuyển sang trạng thái tiếp theo
        state = next_state

## Applying the Learned Policy

After training, we apply the learned policy greedily.

In [5]:
# Áp dụng policy đã học
state = 0
trajectory = [state]  # Lưu lại đường đi của agent
actions = [] # Lưu chuỗi các hành động mà agent thực hiện

while state != terminal_state:
    # Chọn hành động có Q lớn nhất
    q_left = q_value(state, 0)
    q_right = q_value(state, 1)
    action = 0 if q_left >= q_right else 1
    
    # Thực hiện hành động
    state, _ = step(state, action)
    trajectory.append(state)
    actions.append(action)

print("Agent trajectory:", trajectory)  # In đường đi của agent
print("Sequence of actions:", actions) # In chuỗi các hành động của agent

Agent trajectory: [0, 1, 2]
Sequence of actions: [1, 1]


## Conclusion

This example shows how Approximate Q-Learning works using a linear function approximator ($Q(s, a) = w · f(s, a)$).
Instead of storing a full Q-table, we learn a weight vector that generalizes across states and actions.